# 05 · Looking inside

1. **Linear probes** per layer: can the activations predict whether the model will cave?
2. **Steering direction** before and after DPO: is it gone, or only used less?
3. **SAE features** (Gemma 2 2B with Gemma Scope) that rise on sycophantic answers.

Run notebook 04 first; it needs `adapters/{TAG}-dpo`.

## Setup

Runtime → Change runtime type → **T4 GPU**. The repo is cloned to `/content/sycophancy` and results are written to
Google Drive so they survive a disconnect.

In [ ]:
import os
if not os.path.exists("/content/sycophancy"):
    !git clone https://github.com/namita-tj/sycophancy /content/sycophancy
%cd /content/sycophancy
!pip install -q -e ".[all]"

In [ ]:
# Gated models (Llama 3.2, Gemma 2) need a Hugging Face token with access accepted on the model page.
from huggingface_hub import notebook_login
notebook_login()

In [ ]:
# Keep results on Drive.
from google.colab import drive
drive.mount("/content/drive")
RESULTS = "/content/drive/MyDrive/sycophancy/results"
os.makedirs(RESULTS, exist_ok=True)
!rm -rf results && ln -s $RESULTS results

In [ ]:
# Evaluation data (MMLU + Perez et al.) and synthetic training data. Only needed once per runtime.
if not os.path.exists("data/eval/pressure.jsonl"):
    !bash scripts/prepare_data.sh
!wc -l data/eval/*.jsonl data/train/*.jsonl

In [ ]:
MODEL = "Qwen/Qwen2.5-1.5B-Instruct"
TAG = MODEL.split("/")[-1].lower()
DTYPE = "fp16"   # a T4 has no bf16; use bf16 on an A100 or L4
LAYERS = "0,4,8,12,16,20,24"
ADAPTERS = "/content/drive/MyDrive/sycophancy/adapters"

## 1 · Probes

In [ ]:
!syco probe --model {MODEL} --dtype {DTYPE} --data data/eval --split dev --layers {LAYERS} --out results/{TAG}-probe-base.json
!syco probe --model {MODEL} --adapter {ADAPTERS}/{TAG}-dpo --dtype {DTYPE} --data data/eval --split dev --layers {LAYERS} --out results/{TAG}-probe-dpo.json

In [ ]:
import json
import matplotlib.pyplot as plt
for name in ["base", "dpo"]:
    res = json.load(open(f"results/{TAG}-probe-{name}.json"))
    layers = sorted(int(k) for k in res["layers"])
    plt.plot(layers, [res["layers"][str(l)]["balanced_accuracy"] for l in layers], marker="o", label=name)
plt.axhline(0.5, ls="--", c="grey"); plt.xlabel("layer"); plt.ylabel("balanced accuracy"); plt.legend()

## 2 · Steering direction before and after DPO

In [ ]:
!syco steer build --model {MODEL} --dtype {DTYPE} --data data/eval --layers {LAYERS} --out results/{TAG}-vectors-base.npz
!syco steer build --model {MODEL} --adapter {ADAPTERS}/{TAG}-dpo --dtype {DTYPE} --data data/eval --layers {LAYERS} --out results/{TAG}-vectors-dpo.npz
!syco directions --before results/{TAG}-vectors-base.npz --after results/{TAG}-vectors-dpo.npz

High cosine and a smaller norm after DPO → the direction still exists but is used less (suppressed).
Low cosine → training changed the representation itself.

## 3 · SAE features (Gemma 2 2B + Gemma Scope)

In [ ]:
from syco.models.hf import HFModel
from syco.data.schema import read_mcq
from syco.data.splits import select
from syco.methods.steering import contrastive_pairs
from syco.interp.sae import load_gemma_scope, rank_features

LAYER = 12
gemma = HFModel.load("google/gemma-2-2b-it", dtype="fp16")
sae = load_gemma_scope("gemma-scope-2b-pt-res-canonical", f"layer_{LAYER}/width_16k/canonical")
pressure = select(read_mcq("data/eval/pressure.jsonl"), "dev")[:150]
pairs = contrastive_pairs(pressure)
top = rank_features(gemma, sae, pairs, LAYER, top=20)
top

Look up the top features on [Neuronpedia](https://www.neuronpedia.org/gemma-2-2b) to see what they respond to.